# Experiment 7 — Digital Signature Standard (DSS / DSA)
Message authentication and integrity verification

## Math Helpers

In [ ]:
import hashlib, random, math, secrets

def ext_gcd(a,b):
    if a==0: return b,0,1
    g,x,y=ext_gcd(b%a,a); return g,y-(b//a)*x,x

def mod_inv(a,m):
    _,x,_=ext_gcd(a%m,m); return x%m

def miller_rabin(n,k=10):
    if n<2: return False
    if n in(2,3): return True
    if n%2==0: return False
    r,d=0,n-1
    while d%2==0: r+=1; d//=2
    for _ in range(k):
        a=random.randrange(2,n-1); x=pow(a,d,n)
        if x in(1,n-1): continue
        for _ in range(r-1):
            x=pow(x,2,n)
            if x==n-1: break
        else: return False
    return True

print("Helpers defined ✓")


## DSA Parameter & Key Generation

In [ ]:
def dsa_params(L=512,N=160):
    while True:
        q=secrets.randbits(N); q|=(1<<(N-1))|1
        if not miller_rabin(q): continue
        k=secrets.randbits(L-N); p=k*q+1
        if p.bit_length()==L and miller_rabin(p): break
    while True:
        h=random.randint(2,p-2); g=pow(h,(p-1)//q,p)
        if g>1: break
    return p,q,g

def dsa_keygen(p,q,g):
    x=random.randint(1,q-1); y=pow(g,x,p); return x,y

print("Generating DSA parameters (L=512, N=160)...")
p,q,g=dsa_params()
x,y=dsa_keygen(p,q,g)
print(f"p ({p.bit_length()} bits) = {p}")
print(f"q ({q.bit_length()} bits) = {q}")
print(f"g = {g}")
print(f"Private key x = {x}")
print(f"Public  key y = {y}")


## Sign

In [ ]:
def dsa_sign(msg,p,q,g,x):
    h=int(hashlib.sha1(msg).hexdigest(),16)
    while True:
        k=random.randint(1,q-1); r=pow(g,k,p)%q
        if r==0: continue
        s=(mod_inv(k,q)*(h+x*r))%q
        if s!=0: break
    return r,s

msg=b"Hello, this is a secure message!"
r,s=dsa_sign(msg,p,q,g,x)
print(f"Message   : {msg.decode()}")
print(f"SHA-1 H   : {int(hashlib.sha1(msg).hexdigest(),16)}")
print(f"r = {r}")
print(f"s = {s}")


## Verify

In [ ]:
def dsa_verify(msg,sig,p,q,g,y):
    r,s=sig
    if not(0<r<q and 0<s<q): return False
    h=int(hashlib.sha1(msg).hexdigest(),16)
    w=mod_inv(s,q); u1=(h*w)%q; u2=(r*w)%q
    v=(pow(g,u1,p)*pow(y,u2,p))%p%q
    return v==r

valid=dsa_verify(msg,(r,s),p,q,g,y)
print(f"Signature valid (original) : {valid}")
assert valid
print("✓ Authentic signature verified")


## Integrity Tests

In [ ]:
# Tampered message
tampered=b"Hello, this is a TAMPERED message!"
v2=dsa_verify(tampered,(r,s),p,q,g,y)
print(f"Tampered message valid : {v2}")
assert not v2
print("✓ Tampered message correctly rejected")

# Wrong public key
x2,y2=dsa_keygen(p,q,g)
v3=dsa_verify(msg,(r,s),p,q,g,y2)
print(f"Wrong key valid        : {v3}")
assert not v3
print("✓ Wrong key correctly rejected")

print()
print("Security properties:")
print("1. Unforgeability  — only holder of x can produce valid (r,s)")
print("2. Non-repudiation — signer cannot deny producing the signature")
print("3. Integrity       — any bit-flip in message invalidates signature")
print("4. k reuse attack  — reusing k leaks x (critical vulnerability!)")
